# CC3092 — Laboratorio #5
## Agentes en el Arcade Learning Environment (ALE): Space Invaders
## Sebastian Garcia 22291

**Objetivo.** Construir infraestructura reutilizable para que agentes sin entrenamiento interactúen con Gymnasium/ALE y graben episodios completos.

## 1. Arcade Learning Environment

ALE es una plataforma para evaluar agentes generales en juegos de Atari 2600 mediante una interfaz uniforme de aprendizaje por refuerzo. Separa el diseño del agente de la emulación, extrae automáticamente puntuación y fin de partida y permite comparar métodos bajo protocolos reproducibles. Está construido sobre **Stella**, que emula el hardware de Atari 2600 y ejecuta las ROM de los juegos; ALE añade sobre esa emulación la interfaz de observaciones, acciones y recompensas. [Repositorio y documentación oficial de ALE](https://github.com/Farama-Foundation/Arcade-Learning-Environment) y [artículo original de Bellemare et al. (2013)](https://jair.org/index.php/jair/article/view/10819).

### Variantes y parámetros

`ALE/SpaceInvaders-v5` produce RGB, repite cada acción durante 4 frames y usa acciones pegajosas con probabilidad 0.25. En la nomenclatura antigua, el sufijo `-ram` seleccionaba los 128 bytes de RAM; en la API actual también se puede indicar `obs_type="ram"`. `frameskip=n` mantiene una acción durante `n` frames emulados (una tupla elige aleatoriamente en ese intervalo); `repeat_action_probability` es la probabilidad de ejecutar la acción anterior en vez de la solicitada; y `full_action_space=True` expone las 18 combinaciones legales de la consola, mientras `False` usa solo el conjunto mínimo útil del juego. v5 restauró la pegajosidad y eliminó el frame skip aleatorio de v0. [Variantes oficiales de Space Invaders](https://ale.farama.org/main/environments/space_invaders/) y [argumentos comunes de ALE](https://ale.farama.org/environments/).

El *frame skipping* reduce la frecuencia de decisión y, por tanto, el número de inferencias y observaciones que deben procesarse: con salto 4 se decide una vez por cada cuatro frames. Esto acelera simulación y aprendizaje, y hace persistentes las acciones. Un valor excesivo puede omitir eventos breves o reducir el control fino; uno pequeño conserva detalle temporal pero aumenta el costo y produce transiciones muy correlacionadas. El valor debe reportarse para que una comparación sea justa.

### Space Invaders y recompensa

El jugador mueve horizontalmente un cañón, dispara a 36 invasores y evita sus bombas. Gana si elimina la formación antes de que llegue al suelo; pierde una vida al ser alcanzado y la partida termina al agotar las vidas o cuando invaden la base. En el cartucho Atari 2600 los invasores de las seis filas valen 5, 10, 15, 20, 25 y 30 puntos, y la nave de mando normalmente vale 200. [Manual original archivado](https://atariage.com/2600/manuals_old/space_invaders.html). ALE devuelve como recompensa los puntos obtenidos por las acciones simuladas en cada `step`; por eso el retorno acumulado coincide con la puntuación ganada durante el episodio (salvo transformaciones posteriores de recompensa). [Descripción de recompensa de ALE](https://ale.farama.org/main/environments/space_invaders/).

## 2. Espacios de observación y acción

La observación RGB predeterminada es `Box(0, 255, (210, 160, 3), uint8)`: 100,800 intensidades por frame. `CartPole-v1`, en cambio, entrega un `Box` de cuatro números `float32` (posición y velocidad del carro, ángulo y velocidad angular del poste). Los píxeles no identifican explícitamente objetos ni velocidades: elevan dimensionalidad, memoria y costo, suelen requerir una CNN y una secuencia de frames para inferir movimiento. CartPole ya proporciona variables físicas compactas y útiles.

Con `obs_type="ram"`, la observación es `Box(0, 255, (128,), uint8)`. Puede preferirse para prototipos rápidos, menor consumo o experimentos sobre representación sin visión. Su desventaja es que las direcciones de memoria y su significado dependen del juego; los píxeles son una interfaz más general y semejante a lo que ve una persona. [Espacios oficiales de ALE](https://ale.farama.org/environments/).

El conjunto mínimo de Space Invaders es `Discrete(6)`: `0 NOOP` (no hacer nada), `1 FIRE` (disparar), `2 RIGHT`, `3 LEFT`, `4 RIGHTFIRE` y `5 LEFTFIRE`. No conviene codificar estos índices a ciegas: el agente del módulo consulta `get_action_meanings()`. [Tabla oficial de acciones](https://ale.farama.org/main/environments/space_invaders/).

`AtariPreprocessing` implementa reset con 0–30 NOOP aleatorios, frame skip (4), máximo por píxel de los dos frames recientes para mitigar parpadeo, escala de grises, redimensionamiento a 84×84 y, opcionalmente, terminar al perder una vida o normalizar píxeles. **No recorta recompensas**: el *reward clipping* a {-1, 0, 1} pertenece a ciertas canalizaciones de agentes y requeriría un wrapper de recompensa aparte. Además, el entorno base debe crearse con `frameskip=1` para no saltar dos veces. `FrameStackObservation` apila las últimas N observaciones; con cuatro frames permite inferir dirección y velocidad que una imagen aislada no contiene. [AtariPreprocessing](https://gymnasium.farama.org/api/wrappers/misc_wrappers/) y [FrameStackObservation](https://gymnasium.farama.org/api/wrappers/observation_wrappers/).

In [1]:
from pathlib import Path

import gymnasium as gym
import ale_py

from agentes_ale import (
    agente_aleatorio, agente_regla_simple, crear_entorno,
    ejecutar_episodio, generar_video_agente,
)

gym.register_envs(ale_py)
print("Gymnasium:", gym.__version__)
print("ale-py:", ale_py.__version__)

Gymnasium: 1.3.0
ale-py: 0.12.1


## 3. Comprobación de espacios

Se consulta el entorno en lugar de depender solamente de valores escritos en el informe.

In [2]:
env = crear_entorno("ALE/SpaceInvaders-v5")
try:
    obs, info = env.reset(seed=3092)
    print("Observación:", env.observation_space)
    print("Forma/dtype real:", obs.shape, obs.dtype)
    print("Acciones:", env.action_space)
    print("Significados:", env.unwrapped.get_action_meanings())
finally:
    env.close()

Observación: Box(0, 255, (210, 160, 3), uint8)
Forma/dtype real: (210, 160, 3) uint8
Acciones: Discrete(6)
Significados: ['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']


## 4. Módulo reutilizable y agentes

Las cinco funciones solicitadas están implementadas en `agentes_ale.py`. `crear_entorno` funciona con cualquier id de Gymnasium y activa `render_mode="rgb_array"` cuando graba. `ejecutar_episodio` respeta `terminated`, `truncated` y un límite defensivo. `generar_video_agente` usa `try/finally` para cerrar el entorno y devuelve únicamente los MP4 creados en esa llamada, junto con métricas.

El agente de regla simple detecta aproximadamente el cañón y el centro horizontal de los invasores a partir de la imagen; elige `LEFTFIRE`, `RIGHTFIRE` o `FIRE`. Es solo una heurística interpretable, no un agente entrenado. Si no recibe una imagen o acciones ALE, usa una acción válida aleatoria como respaldo.

In [3]:
# Prueba corta de infraestructura sin grabar (no sustituye al episodio completo).
env = crear_entorno("ALE/SpaceInvaders-v5")
try:
    prueba = ejecutar_episodio(env, agente_regla_simple, max_steps=100)
finally:
    env.close()
prueba

{'pasos': 100,
 'recompensa_total': 0.0,
 'terminated': False,
 'truncated': False,
 'limite_alcanzado': True}

## 5. Video del agente aleatorio

La siguiente celda graba **un episodio completo**, cierra el codificador y reporta pasos y retorno. Los resultados concretos quedan guardados en la salida al ejecutar el notebook. `max_steps=10000` es el límite de seguridad solicitado; la métrica `limite_alcanzado` permite distinguirlo de un final natural.

In [4]:
videos, metricas = generar_video_agente(
    nombre_entorno="ALE/SpaceInvaders-v5",
    funcion_agente=agente_aleatorio,
    video_folder=Path("entregables/videos"),
    name_prefix="space-invaders-aleatorio",
    n_episodios=1,
    max_steps=10_000,
)

for numero, metrica in enumerate(metricas, start=1):
    print(
        f"Episodio {numero}: {metrica['pasos']} pasos, "
        f"recompensa total = {metrica['recompensa_total']:.1f}"
    )
print("Videos generados:", videos)

c:\Users\sebas\OneDrive\Escritorio\Github\Deep learning\Laboratorio_5_ALE\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at c:\Users\sebas\OneDrive\Escritorio\Github\Deep learning\Laboratorio_5_ALE\entregables\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episodio 1: 747 pasos, recompensa total = 135.0
Videos generados: ['C:\\Users\\sebas\\OneDrive\\Escritorio\\Github\\Deep learning\\Laboratorio_5_ALE\\entregables\\videos\\space-invaders-aleatorio-episode-0.mp4']


## Referencias

1. Bellemare, M. G. et al. (2013). [The Arcade Learning Environment: An Evaluation Platform for General Agents](https://jair.org/index.php/jair/article/view/10819). *JAIR, 47*, 253–279.
2. Machado, M. C. et al. (2018). [Revisiting the Arcade Learning Environment](https://jair.org/index.php/jair/article/view/11182). *JAIR, 61*, 523–562.
3. Farama Foundation. [Arcade Learning Environment — Environments](https://ale.farama.org/environments/) y [Space Invaders](https://ale.farama.org/main/environments/space_invaders/).
4. Farama Foundation. [Gymnasium wrappers](https://gymnasium.farama.org/api/wrappers/).
5. Atari, Inc. (1980). [Manual de Space Invaders para Atari 2600](https://atariage.com/2600/manuals_old/space_invaders.html).